In [1]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
import sys
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--load_pretrained_model', type=str, default=None,
                    help="预训练模型文件 (.pth) 的路径。如果提供，则加载权重。")
    parser.add_argument('--freeze_backbone', action='store_true',
                    help="如果加载预训练模型，是否冻结骨干网络参数（除分类头外）。")
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    # 首先，根据模型名称和参数实例化模型
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}, device={args.device}")
        model = Lstm(
            factors=num_features,
            batch_size=args.batch_size,
            drop_ratio=args.drop_ratio,
            device=args.device
        )
    elif args.model_name == "Transformer":
        print(f"初始化 Transformer模型: input_dim={num_features}, model_dim={args.model_dim}, depth={args.depth}, num_heads={args.num_heads}, drop_ratio={args.drop_ratio}")
        # 确保 Transformer 构造函数与你的 Transformer.py 一致
        model = Transformer(
            input_dim=num_features,
            model_dim=args.model_dim,
            depth=args.depth,
            num_heads=args.num_heads,
            drop_ratio=args.drop_ratio
            # qkv_bias=True, # 根据你的Transformer.py添加其他必要参数
            # attn_drop_ratio=args.drop_ratio,
            # drop_path_ratio=args.drop_path_ratio if hasattr(args, 'drop_path_ratio') else 0.1,
        )
    elif args.model_name == "GRU":
        print(f"初始化 GRU模型: factors={num_features}, batch_size={args.batch_size}, num_layers={args.num_layers}, drop_ratio={args.drop_ratio}, device={args.device}")
        model = GRU(
            factors=num_features,
            batch_size=args.batch_size,
            num_layers=args.num_layers,
            drop_ratio=args.drop_ratio,
            device=args.device
        )
    elif args.model_name == "GPT":
        # 严格遵循原始 train.py 中 GPT 的实例化逻辑
        gpt_actual_input_dim = num_features
        gpt_internal_hidden_dim = num_features # 模型内部工作维度，为了匹配原始 hidden_dim=factors
        gpt_internal_num_heads = 4           # 硬编码自原始 train.py
        gpt_internal_ff_expansion_factor = 128 # 硬编码自原始 train.py

        print(f"初始化 GPT模型 (尝试严格复现原始 train.py 实例化):")
        print(f"  actual_input_dim: {gpt_actual_input_dim}")
        print(f"  hidden_dim: {gpt_internal_hidden_dim}")
        print(f"  num_heads: {gpt_internal_num_heads}")
        print(f"  num_layers: {args.num_layers}")
        print(f"  dropout: {args.drop_ratio}")
        print(f"  ff_expansion_factor: {gpt_internal_ff_expansion_factor}")

        # 确保你的 model_gpt.py 中的 GPT 类 __init__ 签名是这样的：
        # def __init__(self, actual_input_dim, hidden_dim, num_heads, num_layers,
        #              ff_expansion_factor, dropout, max_seq_len=512, device='cuda'):
        # 并且它内部正确使用了 actual_input_dim 来创建一个投影层到 hidden_dim
        model = GPT(
            actual_input_dim=gpt_actual_input_dim,
            hidden_dim=gpt_internal_hidden_dim,
            num_heads=gpt_internal_num_heads,
            num_layers=args.num_layers,
            ff_expansion_factor=gpt_internal_ff_expansion_factor,
            dropout=args.drop_ratio,
            device=args.device
            # max_seq_len 如果需要，也应传递
        )
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")

    # 将模型移动到设备
    model = model.to(args.device)

    # --- 加载预训练权重逻辑 ---
    if args.load_pretrained_model and hasattr(args, 'load_pretrained_model') and args.load_pretrained_model is not None: # 检查参数是否存在且非None
        if os.path.exists(args.load_pretrained_model):
            print(f"\n加载预训练权重从: {args.load_pretrained_model}")
            try:
                pretrained_state_dict = torch.load(args.load_pretrained_model, map_location=args.device)

                # 处理常见的state_dict包装方式
                if isinstance(pretrained_state_dict, dict):
                    if 'state_dict' in pretrained_state_dict:
                        pretrained_state_dict = pretrained_state_dict['state_dict']
                    elif 'model_state_dict' in pretrained_state_dict:
                        pretrained_state_dict = pretrained_state_dict['model_state_dict']
                    # 如果直接就是 state_dict，则无需处理

                current_model_dict = model.state_dict()

                # --- 确定分类头的名称 ---
                # 这需要根据你每个模型的具体实现来确定
                # 假设 Transformer 的分类头是 'MLP' (基于你原始的 Transformer.py 中的 self.MLP)
                # 假设 Lstm, GRU, GPT 的分类头都叫做 'head' (基于常见的命名)
                # 你需要根据实际情况修改这些名称！
                classifier_head_keyword = None
                if args.model_name == "Transformer":
                    classifier_head_keyword = "MLP." # 加点号确保是模块名的一部分
                elif args.model_name in ["Lstm", "GRU", "GPT"]:
                    # 检查你的模型定义中分类头的真实名称
                    # 假设在这些模型中，最终的线性层或包含最终线性层的Sequential块叫做 'head'
                    if hasattr(model, 'head'):
                        classifier_head_keyword = "head."
                    else:
                        print(f"警告: 模型 {args.model_name} 没有找到名为 'head' 的属性，无法准确排除分类头权重。请检查模型定义。")
                
                if classifier_head_keyword is None and args.model_name != "Transformer": # Transformer 有特例
                     print(f"警告: 未能确定模型 {args.model_name} 的分类头名称，将尝试加载所有匹配的权重。这可能导致分类头也被预训练权重覆盖。")


                state_dict_to_load = {}
                weights_loaded_count = 0
                weights_skipped_shape_mismatch = 0
                weights_skipped_classifier = 0

                for k, v_pretrained in pretrained_state_dict.items():
                    load_this_weight = True
                    if classifier_head_keyword: # 如果我们定义了分类头的关键词
                        if k.startswith(classifier_head_keyword):
                            weights_skipped_classifier += 1
                            load_this_weight = False # 不加载分类头的权重
                    
                    if load_this_weight:
                        if k in current_model_dict:
                            v_current = current_model_dict[k]
                            if v_current.shape == v_pretrained.shape:
                                state_dict_to_load[k] = v_pretrained
                                weights_loaded_count += 1
                            else:
                                print(f"  形状不匹配，跳过加载权重 {k}: 预训练 {v_pretrained.shape}, 当前 {v_current.shape}")
                                weights_skipped_shape_mismatch += 1
                        # else: # 预训练模型中有，但当前模型没有的键，通常可以忽略
                        #     print(f"  权重 {k} 在预训练模型中存在，但在当前模型中不存在，跳过。")
                
                if not state_dict_to_load:
                    print("警告: 没有从预训练模型中找到可加载的匹配权重（非分类头部分）。")
                else:
                    print(f"将从预训练模型加载 {weights_loaded_count} 个参数组。")
                    if weights_skipped_classifier > 0:
                        print(f"  跳过了 {weights_skipped_classifier} 个分类头相关的参数组。")
                    if weights_skipped_shape_mismatch > 0:
                        print(f"  因形状不匹配跳过了 {weights_skipped_shape_mismatch} 个参数组。")
                    
                    current_model_dict.update(state_dict_to_load)
                    # 使用 strict=False 允许当前模型中存在预训练模型没有的键（例如新初始化的分类头）
                    # 也允许预训练模型中存在当前模型没有的键（虽然我们已经通过 k in current_model_dict 过滤了一部分）
                    model.load_state_dict(current_model_dict, strict=False)
                    print("预训练权重加载完成。分类头（如果存在且未被加载）保持其原有初始化。")

                # --- 冻结骨干网络参数 ---
                if args.freeze_backbone and hasattr(args, 'freeze_backbone') and args.freeze_backbone:
                    print("冻结骨干网络参数...")
                    num_frozen = 0
                    num_trainable = 0
                    for name, param in model.named_parameters():
                        freeze_this_param = True
                        if classifier_head_keyword: # 如果我们定义了分类头的关键词
                            if name.startswith(classifier_head_keyword):
                                freeze_this_param = False # 不冻结分类头的参数
                        
                        if freeze_this_param:
                            param.requires_grad = False
                            num_frozen +=1
                        else:
                            param.requires_grad = True # 确保分类头是可训练的
                            print(f"  分类头参数 '{name}' 将保持可训练。")
                            num_trainable +=1
                    print(f"  已冻结 {num_frozen} 组参数。")
                    print(f"  保持可训练 {num_trainable} 组参数 (主要是分类头)。")
            except Exception as e:
                print(f"错误: 加载预训练权重时发生异常: {e}")
                print("将从头开始训练模型。")
        else:
            print(f"警告: 预训练模型文件未找到: {args.load_pretrained_model}。将从头开始训练。")
    else:
        print("未提供预训练模型路径，或未启用预训练加载。模型将从头开始训练。")
        
    return model

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")



import sys # 确保导入sys模块



if __name__ == '__main__':
    # --- 模拟命令行参数: 测试 Transformer 在 8因子 MIMIC-IV 数据上 (带预训练) ---
    PRETRAINED_MODEL_PATH = "/mnt/public/home/zhijiangwan/sepsisformer/sepsisformer/model2/dl_model_outputs_replication/Transformer_mimic3_eICU_8_factors_mimic3eICU-8f-transformer-replication_trainseed42_20250610-041619/best_auc_model.pth" # 你的预训练模型路径

    sys.argv = [
        'my_script_name_in_notebook.py',
        '--presplit_data_dir', './temp_data_utils_logs/mimic4_8_factors_split',
        '--model_name', 'Transformer',
        '--model_dim', '128',
        '--depth', '8',
        '--num_heads', '8',
        '--drop_ratio', '0.1',
        '--lr', '0.001', # 原始摘要的学习率
        '--epochs', '450', # 原始摘要的 epochs
        '--batch_size', '5000',
        '--output_base_dir', './dl_model_outputs_replication',
        '--experiment_tag', 'mimic4-8f-transformer-finetune', # 标签改为 finetune
        '--seed', '42',
        '--device', 'cuda',
        '--label_column', 'dead',
        # --- 新增预训练参数 ---
        '--load_pretrained_model', PRETRAINED_MODEL_PATH,
        # '--freeze_backbone' # 如果需要冻结骨干，取消此行注释
    ]
    main()



--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 450
  experiment_tag: mimic4-8f-transformer-finetune
  freeze_backbone: False
  label_column: dead
  load_pretrained_model: /mnt/public/home/zhijiangwan/sepsisformer/sepsisformer/model2/dl_model_outputs_replication/Transformer_mimic3_eICU_8_factors_mimic3eICU-8f-transformer-replication_trainseed42_20250610-041619/best_auc_model.pth
  lr: 0.001
  model_dim: 128
  model_name: Transformer
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic4_8_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/Transformer_mimic4_8_factors_mimic4-8f-transformer-finetune_trainseed42_20250610-052044
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_test_data_seed256.csv
  - 特征数量: 8
  - 训练样本数: 4525


/tmp/ipykernel_729/2823220024.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_state_dict = torch.load(args.load_pretrained_model, map_location=args.device)


优化器: Adam, LR: 0.001, Weight Decay: 0
损失函数: NMTCritierion (标签平滑: 0.3)

--- 开始训练 ---


/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


  * New best Test AUC: 0.6393 at Epoch 1. Model and DeLong data updated.
Epoch [001/450] | Train Loss: 400.2354 | Test Acc: 0.5923 | Test AUC: 0.6393 | Time: 0.53s
  * New best Test AUC: 0.6511 at Epoch 2. Model and DeLong data updated.
Epoch [002/450] | Train Loss: 355.4488 | Test Acc: 0.6149 | Test AUC: 0.6511 | Time: 0.14s
  * New best Test AUC: 0.6654 at Epoch 3. Model and DeLong data updated.
Epoch [003/450] | Train Loss: 348.8959 | Test Acc: 0.6309 | Test AUC: 0.6654 | Time: 0.27s
  * New best Test AUC: 0.6722 at Epoch 4. Model and DeLong data updated.
Epoch [004/450] | Train Loss: 343.4413 | Test Acc: 0.6345 | Test AUC: 0.6722 | Time: 0.14s
  * New best Test AUC: 0.6743 at Epoch 5. Model and DeLong data updated.
Epoch [005/450] | Train Loss: 341.9659 | Test Acc: 0.6320 | Test AUC: 0.6743 | Time: 0.23s
  * New best Test AUC: 0.6762 at Epoch 6. Model and DeLong data updated.
Epoch [006/450] | Train Loss: 342.2613 | Test Acc: 0.6314 | Test AUC: 0.6762 | Time: 0.16s
  * New best Tes

In [2]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
import sys
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--load_pretrained_model', type=str, default=None,
                    help="预训练模型文件 (.pth) 的路径。如果提供，则加载权重。")
    parser.add_argument('--freeze_backbone', action='store_true',
                    help="如果加载预训练模型，是否冻结骨干网络参数（除分类头外）。")
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    # 首先，根据模型名称和参数实例化模型
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}, device={args.device}")
        model = Lstm(
            factors=num_features,
            batch_size=args.batch_size,
            drop_ratio=args.drop_ratio,
            device=args.device
        )
    elif args.model_name == "Transformer":
        print(f"初始化 Transformer模型: input_dim={num_features}, model_dim={args.model_dim}, depth={args.depth}, num_heads={args.num_heads}, drop_ratio={args.drop_ratio}")
        # 确保 Transformer 构造函数与你的 Transformer.py 一致
        model = Transformer(
            input_dim=num_features,
            model_dim=args.model_dim,
            depth=args.depth,
            num_heads=args.num_heads,
            drop_ratio=args.drop_ratio
            # qkv_bias=True, # 根据你的Transformer.py添加其他必要参数
            # attn_drop_ratio=args.drop_ratio,
            # drop_path_ratio=args.drop_path_ratio if hasattr(args, 'drop_path_ratio') else 0.1,
        )
    elif args.model_name == "GRU":
        print(f"初始化 GRU模型: factors={num_features}, batch_size={args.batch_size}, num_layers={args.num_layers}, drop_ratio={args.drop_ratio}, device={args.device}")
        model = GRU(
            factors=num_features,
            batch_size=args.batch_size,
            num_layers=args.num_layers,
            drop_ratio=args.drop_ratio,
            device=args.device
        )
    elif args.model_name == "GPT":
        # 严格遵循原始 train.py 中 GPT 的实例化逻辑
        gpt_actual_input_dim = num_features
        gpt_internal_hidden_dim = num_features # 模型内部工作维度，为了匹配原始 hidden_dim=factors
        gpt_internal_num_heads = 4           # 硬编码自原始 train.py
        gpt_internal_ff_expansion_factor = 128 # 硬编码自原始 train.py

        print(f"初始化 GPT模型 (尝试严格复现原始 train.py 实例化):")
        print(f"  actual_input_dim: {gpt_actual_input_dim}")
        print(f"  hidden_dim: {gpt_internal_hidden_dim}")
        print(f"  num_heads: {gpt_internal_num_heads}")
        print(f"  num_layers: {args.num_layers}")
        print(f"  dropout: {args.drop_ratio}")
        print(f"  ff_expansion_factor: {gpt_internal_ff_expansion_factor}")

        # 确保你的 model_gpt.py 中的 GPT 类 __init__ 签名是这样的：
        # def __init__(self, actual_input_dim, hidden_dim, num_heads, num_layers,
        #              ff_expansion_factor, dropout, max_seq_len=512, device='cuda'):
        # 并且它内部正确使用了 actual_input_dim 来创建一个投影层到 hidden_dim
        model = GPT(
            actual_input_dim=gpt_actual_input_dim,
            hidden_dim=gpt_internal_hidden_dim,
            num_heads=gpt_internal_num_heads,
            num_layers=args.num_layers,
            ff_expansion_factor=gpt_internal_ff_expansion_factor,
            dropout=args.drop_ratio,
            device=args.device
            # max_seq_len 如果需要，也应传递
        )
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")

    # 将模型移动到设备
    model = model.to(args.device)

    # --- 加载预训练权重逻辑 ---
    if args.load_pretrained_model and hasattr(args, 'load_pretrained_model') and args.load_pretrained_model is not None: # 检查参数是否存在且非None
        if os.path.exists(args.load_pretrained_model):
            print(f"\n加载预训练权重从: {args.load_pretrained_model}")
            try:
                pretrained_state_dict = torch.load(args.load_pretrained_model, map_location=args.device)

                # 处理常见的state_dict包装方式
                if isinstance(pretrained_state_dict, dict):
                    if 'state_dict' in pretrained_state_dict:
                        pretrained_state_dict = pretrained_state_dict['state_dict']
                    elif 'model_state_dict' in pretrained_state_dict:
                        pretrained_state_dict = pretrained_state_dict['model_state_dict']
                    # 如果直接就是 state_dict，则无需处理

                current_model_dict = model.state_dict()

                # --- 确定分类头的名称 ---
                # 这需要根据你每个模型的具体实现来确定
                # 假设 Transformer 的分类头是 'MLP' (基于你原始的 Transformer.py 中的 self.MLP)
                # 假设 Lstm, GRU, GPT 的分类头都叫做 'head' (基于常见的命名)
                # 你需要根据实际情况修改这些名称！
                classifier_head_keyword = None
                if args.model_name == "Transformer":
                    classifier_head_keyword = "MLP." # 加点号确保是模块名的一部分
                elif args.model_name in ["Lstm", "GRU", "GPT"]:
                    # 检查你的模型定义中分类头的真实名称
                    # 假设在这些模型中，最终的线性层或包含最终线性层的Sequential块叫做 'head'
                    if hasattr(model, 'head'):
                        classifier_head_keyword = "head."
                    else:
                        print(f"警告: 模型 {args.model_name} 没有找到名为 'head' 的属性，无法准确排除分类头权重。请检查模型定义。")
                
                if classifier_head_keyword is None and args.model_name != "Transformer": # Transformer 有特例
                     print(f"警告: 未能确定模型 {args.model_name} 的分类头名称，将尝试加载所有匹配的权重。这可能导致分类头也被预训练权重覆盖。")


                state_dict_to_load = {}
                weights_loaded_count = 0
                weights_skipped_shape_mismatch = 0
                weights_skipped_classifier = 0

                for k, v_pretrained in pretrained_state_dict.items():
                    load_this_weight = True
                    if classifier_head_keyword: # 如果我们定义了分类头的关键词
                        if k.startswith(classifier_head_keyword):
                            weights_skipped_classifier += 1
                            load_this_weight = False # 不加载分类头的权重
                    
                    if load_this_weight:
                        if k in current_model_dict:
                            v_current = current_model_dict[k]
                            if v_current.shape == v_pretrained.shape:
                                state_dict_to_load[k] = v_pretrained
                                weights_loaded_count += 1
                            else:
                                print(f"  形状不匹配，跳过加载权重 {k}: 预训练 {v_pretrained.shape}, 当前 {v_current.shape}")
                                weights_skipped_shape_mismatch += 1
                        # else: # 预训练模型中有，但当前模型没有的键，通常可以忽略
                        #     print(f"  权重 {k} 在预训练模型中存在，但在当前模型中不存在，跳过。")
                
                if not state_dict_to_load:
                    print("警告: 没有从预训练模型中找到可加载的匹配权重（非分类头部分）。")
                else:
                    print(f"将从预训练模型加载 {weights_loaded_count} 个参数组。")
                    if weights_skipped_classifier > 0:
                        print(f"  跳过了 {weights_skipped_classifier} 个分类头相关的参数组。")
                    if weights_skipped_shape_mismatch > 0:
                        print(f"  因形状不匹配跳过了 {weights_skipped_shape_mismatch} 个参数组。")
                    
                    current_model_dict.update(state_dict_to_load)
                    # 使用 strict=False 允许当前模型中存在预训练模型没有的键（例如新初始化的分类头）
                    # 也允许预训练模型中存在当前模型没有的键（虽然我们已经通过 k in current_model_dict 过滤了一部分）
                    model.load_state_dict(current_model_dict, strict=False)
                    print("预训练权重加载完成。分类头（如果存在且未被加载）保持其原有初始化。")

                # --- 冻结骨干网络参数 ---
                if args.freeze_backbone and hasattr(args, 'freeze_backbone') and args.freeze_backbone:
                    print("冻结骨干网络参数...")
                    num_frozen = 0
                    num_trainable = 0
                    for name, param in model.named_parameters():
                        freeze_this_param = True
                        if classifier_head_keyword: # 如果我们定义了分类头的关键词
                            if name.startswith(classifier_head_keyword):
                                freeze_this_param = False # 不冻结分类头的参数
                        
                        if freeze_this_param:
                            param.requires_grad = False
                            num_frozen +=1
                        else:
                            param.requires_grad = True # 确保分类头是可训练的
                            print(f"  分类头参数 '{name}' 将保持可训练。")
                            num_trainable +=1
                    print(f"  已冻结 {num_frozen} 组参数。")
                    print(f"  保持可训练 {num_trainable} 组参数 (主要是分类头)。")
            except Exception as e:
                print(f"错误: 加载预训练权重时发生异常: {e}")
                print("将从头开始训练模型。")
        else:
            print(f"警告: 预训练模型文件未找到: {args.load_pretrained_model}。将从头开始训练。")
    else:
        print("未提供预训练模型路径，或未启用预训练加载。模型将从头开始训练。")
        
    return model

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")



import sys # 确保导入sys模块



if __name__ == '__main__':
    # --- 模拟命令行参数: 测试 Transformer 在 8因子 MIMIC-IV 数据上 (带预训练) ---
    PRETRAINED_LSTM_MODEL_PATH = "./dl_model_outputs_replication/Lstm_mimic3_eICU_8_factors_mimic3eICU-8f-lstm-replication_trainseed42_20250610-043211/best_auc_model.pth"

    # --- 模拟命令行参数: 测试 Lstm 在 8因子 MIMIC-IV 数据上 (带预训练) ---
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic4_8_factors_split', # 8因子 MIMIC-IV 预分割数据目录
        '--model_name', 'Lstm',                                              # 模型名称
        # --- Lstm 相关参数 ---
        '--drop_ratio', '0.1',      # Dropout 比率
        '--num_layers', '2',        # 命令行参数，但你的Lstm实现可能不使用它
        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.0005',           # 学习率
        '--epochs', '1600',         # Epochs
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 输出基础目录
        '--experiment_tag', 'mimic4-8f-lstm-finetune',       # 实验标签，指明是微调
        '--seed', '42',             # 随机种子
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead',   # 标签列名
        # --- 新增预训练参数 ---
        '--load_pretrained_model', PRETRAINED_LSTM_MODEL_PATH,
        # '--freeze_backbone' # 如果需要冻结骨干，取消此行注释 (action='store_true'参数不需要值)
    ]
    main()


--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 1600
  experiment_tag: mimic4-8f-lstm-finetune
  freeze_backbone: False
  label_column: dead
  load_pretrained_model: ./dl_model_outputs_replication/Lstm_mimic3_eICU_8_factors_mimic3eICU-8f-lstm-replication_trainseed42_20250610-043211/best_auc_model.pth
  lr: 0.0005
  model_dim: 128
  model_name: Lstm
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic4_8_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/Lstm_mimic4_8_factors_mimic4-8f-lstm-finetune_trainseed42_20250610-053609
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_test_data_seed256.csv
  - 特征数量: 8
  - 训练样本数: 4525
  - 测试样本数: 1940
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic4_8_factors_split，特征数: 8
初始化 Lstm模型: facto

/tmp/ipykernel_729/3576374374.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_state_dict = torch.load(args.load_pretrained_model, map_location=args.device)


  * New best Test AUC: 0.6149 at Epoch 1. Model and DeLong data updated.
Epoch [001/1600] | Train Loss: 371.7787 | Test Acc: 0.5443 | Test AUC: 0.6149 | Time: 0.31s
  * New best Test AUC: 0.6303 at Epoch 2. Model and DeLong data updated.
Epoch [002/1600] | Train Loss: 364.5112 | Test Acc: 0.5675 | Test AUC: 0.6303 | Time: 0.14s
  * New best Test AUC: 0.6398 at Epoch 3. Model and DeLong data updated.
Epoch [003/1600] | Train Loss: 358.4674 | Test Acc: 0.5928 | Test AUC: 0.6398 | Time: 0.14s
  * New best Test AUC: 0.6486 at Epoch 4. Model and DeLong data updated.
Epoch [004/1600] | Train Loss: 353.6285 | Test Acc: 0.6052 | Test AUC: 0.6486 | Time: 0.14s
  * New best Test AUC: 0.6544 at Epoch 5. Model and DeLong data updated.
Epoch [005/1600] | Train Loss: 348.2256 | Test Acc: 0.6139 | Test AUC: 0.6544 | Time: 0.19s
  * New best Test AUC: 0.6601 at Epoch 6. Model and DeLong data updated.
Epoch [006/1600] | Train Loss: 346.1759 | Test Acc: 0.6134 | Test AUC: 0.6601 | Time: 0.18s
  * New be

In [3]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
import sys
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--load_pretrained_model', type=str, default=None,
                    help="预训练模型文件 (.pth) 的路径。如果提供，则加载权重。")
    parser.add_argument('--freeze_backbone', action='store_true',
                    help="如果加载预训练模型，是否冻结骨干网络参数（除分类头外）。")
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    # 首先，根据模型名称和参数实例化模型
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}, device={args.device}")
        model = Lstm(
            factors=num_features,
            batch_size=args.batch_size,
            drop_ratio=args.drop_ratio,
            device=args.device
        )
    elif args.model_name == "Transformer":
        print(f"初始化 Transformer模型: input_dim={num_features}, model_dim={args.model_dim}, depth={args.depth}, num_heads={args.num_heads}, drop_ratio={args.drop_ratio}")
        # 确保 Transformer 构造函数与你的 Transformer.py 一致
        model = Transformer(
            input_dim=num_features,
            model_dim=args.model_dim,
            depth=args.depth,
            num_heads=args.num_heads,
            drop_ratio=args.drop_ratio
            # qkv_bias=True, # 根据你的Transformer.py添加其他必要参数
            # attn_drop_ratio=args.drop_ratio,
            # drop_path_ratio=args.drop_path_ratio if hasattr(args, 'drop_path_ratio') else 0.1,
        )
    elif args.model_name == "GRU":
        print(f"初始化 GRU模型: factors={num_features}, batch_size={args.batch_size}, num_layers={args.num_layers}, drop_ratio={args.drop_ratio}, device={args.device}")
        model = GRU(
            factors=num_features,
            batch_size=args.batch_size,
            num_layers=args.num_layers,
            drop_ratio=args.drop_ratio,
            device=args.device
        )
    elif args.model_name == "GPT":
        # 严格遵循原始 train.py 中 GPT 的实例化逻辑
        gpt_actual_input_dim = num_features
        gpt_internal_hidden_dim = num_features # 模型内部工作维度，为了匹配原始 hidden_dim=factors
        gpt_internal_num_heads = 4           # 硬编码自原始 train.py
        gpt_internal_ff_expansion_factor = 128 # 硬编码自原始 train.py

        print(f"初始化 GPT模型 (尝试严格复现原始 train.py 实例化):")
        print(f"  actual_input_dim: {gpt_actual_input_dim}")
        print(f"  hidden_dim: {gpt_internal_hidden_dim}")
        print(f"  num_heads: {gpt_internal_num_heads}")
        print(f"  num_layers: {args.num_layers}")
        print(f"  dropout: {args.drop_ratio}")
        print(f"  ff_expansion_factor: {gpt_internal_ff_expansion_factor}")

        # 确保你的 model_gpt.py 中的 GPT 类 __init__ 签名是这样的：
        # def __init__(self, actual_input_dim, hidden_dim, num_heads, num_layers,
        #              ff_expansion_factor, dropout, max_seq_len=512, device='cuda'):
        # 并且它内部正确使用了 actual_input_dim 来创建一个投影层到 hidden_dim
        model = GPT(
            actual_input_dim=gpt_actual_input_dim,
            hidden_dim=gpt_internal_hidden_dim,
            num_heads=gpt_internal_num_heads,
            num_layers=args.num_layers,
            ff_expansion_factor=gpt_internal_ff_expansion_factor,
            dropout=args.drop_ratio,
            device=args.device
            # max_seq_len 如果需要，也应传递
        )
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")

    # 将模型移动到设备
    model = model.to(args.device)

    # --- 加载预训练权重逻辑 ---
    if args.load_pretrained_model and hasattr(args, 'load_pretrained_model') and args.load_pretrained_model is not None: # 检查参数是否存在且非None
        if os.path.exists(args.load_pretrained_model):
            print(f"\n加载预训练权重从: {args.load_pretrained_model}")
            try:
                pretrained_state_dict = torch.load(args.load_pretrained_model, map_location=args.device)

                # 处理常见的state_dict包装方式
                if isinstance(pretrained_state_dict, dict):
                    if 'state_dict' in pretrained_state_dict:
                        pretrained_state_dict = pretrained_state_dict['state_dict']
                    elif 'model_state_dict' in pretrained_state_dict:
                        pretrained_state_dict = pretrained_state_dict['model_state_dict']
                    # 如果直接就是 state_dict，则无需处理

                current_model_dict = model.state_dict()

                # --- 确定分类头的名称 ---
                # 这需要根据你每个模型的具体实现来确定
                # 假设 Transformer 的分类头是 'MLP' (基于你原始的 Transformer.py 中的 self.MLP)
                # 假设 Lstm, GRU, GPT 的分类头都叫做 'head' (基于常见的命名)
                # 你需要根据实际情况修改这些名称！
                classifier_head_keyword = None
                if args.model_name == "Transformer":
                    classifier_head_keyword = "MLP." # 加点号确保是模块名的一部分
                elif args.model_name in ["Lstm", "GRU", "GPT"]:
                    # 检查你的模型定义中分类头的真实名称
                    # 假设在这些模型中，最终的线性层或包含最终线性层的Sequential块叫做 'head'
                    if hasattr(model, 'head'):
                        classifier_head_keyword = "head."
                    else:
                        print(f"警告: 模型 {args.model_name} 没有找到名为 'head' 的属性，无法准确排除分类头权重。请检查模型定义。")
                
                if classifier_head_keyword is None and args.model_name != "Transformer": # Transformer 有特例
                     print(f"警告: 未能确定模型 {args.model_name} 的分类头名称，将尝试加载所有匹配的权重。这可能导致分类头也被预训练权重覆盖。")


                state_dict_to_load = {}
                weights_loaded_count = 0
                weights_skipped_shape_mismatch = 0
                weights_skipped_classifier = 0

                for k, v_pretrained in pretrained_state_dict.items():
                    load_this_weight = True
                    if classifier_head_keyword: # 如果我们定义了分类头的关键词
                        if k.startswith(classifier_head_keyword):
                            weights_skipped_classifier += 1
                            load_this_weight = False # 不加载分类头的权重
                    
                    if load_this_weight:
                        if k in current_model_dict:
                            v_current = current_model_dict[k]
                            if v_current.shape == v_pretrained.shape:
                                state_dict_to_load[k] = v_pretrained
                                weights_loaded_count += 1
                            else:
                                print(f"  形状不匹配，跳过加载权重 {k}: 预训练 {v_pretrained.shape}, 当前 {v_current.shape}")
                                weights_skipped_shape_mismatch += 1
                        # else: # 预训练模型中有，但当前模型没有的键，通常可以忽略
                        #     print(f"  权重 {k} 在预训练模型中存在，但在当前模型中不存在，跳过。")
                
                if not state_dict_to_load:
                    print("警告: 没有从预训练模型中找到可加载的匹配权重（非分类头部分）。")
                else:
                    print(f"将从预训练模型加载 {weights_loaded_count} 个参数组。")
                    if weights_skipped_classifier > 0:
                        print(f"  跳过了 {weights_skipped_classifier} 个分类头相关的参数组。")
                    if weights_skipped_shape_mismatch > 0:
                        print(f"  因形状不匹配跳过了 {weights_skipped_shape_mismatch} 个参数组。")
                    
                    current_model_dict.update(state_dict_to_load)
                    # 使用 strict=False 允许当前模型中存在预训练模型没有的键（例如新初始化的分类头）
                    # 也允许预训练模型中存在当前模型没有的键（虽然我们已经通过 k in current_model_dict 过滤了一部分）
                    model.load_state_dict(current_model_dict, strict=False)
                    print("预训练权重加载完成。分类头（如果存在且未被加载）保持其原有初始化。")

                # --- 冻结骨干网络参数 ---
                if args.freeze_backbone and hasattr(args, 'freeze_backbone') and args.freeze_backbone:
                    print("冻结骨干网络参数...")
                    num_frozen = 0
                    num_trainable = 0
                    for name, param in model.named_parameters():
                        freeze_this_param = True
                        if classifier_head_keyword: # 如果我们定义了分类头的关键词
                            if name.startswith(classifier_head_keyword):
                                freeze_this_param = False # 不冻结分类头的参数
                        
                        if freeze_this_param:
                            param.requires_grad = False
                            num_frozen +=1
                        else:
                            param.requires_grad = True # 确保分类头是可训练的
                            print(f"  分类头参数 '{name}' 将保持可训练。")
                            num_trainable +=1
                    print(f"  已冻结 {num_frozen} 组参数。")
                    print(f"  保持可训练 {num_trainable} 组参数 (主要是分类头)。")
            except Exception as e:
                print(f"错误: 加载预训练权重时发生异常: {e}")
                print("将从头开始训练模型。")
        else:
            print(f"警告: 预训练模型文件未找到: {args.load_pretrained_model}。将从头开始训练。")
    else:
        print("未提供预训练模型路径，或未启用预训练加载。模型将从头开始训练。")
        
    return model

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")



import sys # 确保导入sys模块



if __name__ == '__main__':
    # --- 模拟命令行参数: 测试 Transformer 在 8因子 MIMIC-IV 数据上 (带预训练) ---
    PRETRAINED_GRU_MODEL_PATH = "./dl_model_outputs_replication/GRU_mimic3_eICU_8_factors_mimic3eICU-8f-gru-replication_trainseed42_20250610-050443/best_auc_model.pth"

    # --- 模拟命令行参数: 测试 GRU 在 8因子 MIMIC-IV 数据上 (带预训练) ---
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic4_8_factors_split', # 8因子 MIMIC-IV 预分割数据目录
        '--model_name', 'GRU',                                               # 模型名称
        # --- GRU 特定参数 ---
        '--num_layers', '2',        # GRU 的层数
        '--drop_ratio', '0.1',      # Dropout 比率
        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.001',            # 学习率
        '--epochs', '3000',         # Epochs
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 输出基础目录
        '--experiment_tag', 'mimic4-8f-gru-finetune',        # 实验标签，指明是微调
        '--seed', '42',             # 随机种子
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead',   # 标签列名
        # --- 新增预训练参数 ---
        '--load_pretrained_model', PRETRAINED_GRU_MODEL_PATH,
        # '--freeze_backbone' # 如果需要冻结骨干，取消此行注释
    ]
    main()


--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 3000
  experiment_tag: mimic4-8f-gru-finetune
  freeze_backbone: False
  label_column: dead
  load_pretrained_model: ./dl_model_outputs_replication/GRU_mimic3_eICU_8_factors_mimic3eICU-8f-gru-replication_trainseed42_20250610-050443/best_auc_model.pth
  lr: 0.001
  model_dim: 128
  model_name: GRU
  num_heads: 8
  num_layers: 2
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic4_8_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/GRU_mimic4_8_factors_mimic4-8f-gru-finetune_trainseed42_20250610-053959
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_test_data_seed256.csv
  - 特征数量: 8
  - 训练样本数: 4525
  - 测试样本数: 1940
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic4_8_factors_split，特征数: 8
初始化 GRU模型: factors=8, ba

/tmp/ipykernel_729/3307694685.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_state_dict = torch.load(args.load_pretrained_model, map_location=args.device)


  * New best Test AUC: 0.6437 at Epoch 2. Model and DeLong data updated.
Epoch [002/3000] | Train Loss: 351.8512 | Test Acc: 0.4680 | Test AUC: 0.6437 | Time: 0.11s
  * New best Test AUC: 0.6525 at Epoch 3. Model and DeLong data updated.
Epoch [003/3000] | Train Loss: 349.4962 | Test Acc: 0.4680 | Test AUC: 0.6525 | Time: 0.11s
  * New best Test AUC: 0.6596 at Epoch 4. Model and DeLong data updated.
Epoch [004/3000] | Train Loss: 345.5110 | Test Acc: 0.4680 | Test AUC: 0.6596 | Time: 0.11s
  * New best Test AUC: 0.6653 at Epoch 5. Model and DeLong data updated.
Epoch [005/3000] | Train Loss: 342.8687 | Test Acc: 0.4680 | Test AUC: 0.6653 | Time: 0.11s
  * New best Test AUC: 0.6696 at Epoch 6. Model and DeLong data updated.
Epoch [006/3000] | Train Loss: 341.5073 | Test Acc: 0.4680 | Test AUC: 0.6696 | Time: 0.11s
  * New best Test AUC: 0.6738 at Epoch 7. Model and DeLong data updated.
Epoch [007/3000] | Train Loss: 340.7833 | Test Acc: 0.4680 | Test AUC: 0.6738 | Time: 0.11s
  * New be

In [1]:
# main_deep_learning_trainer.py
import os
import argparse
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
import pandas as pd

from Transformer import Transformer # 假设其他模型导入也正确
from model_LSTM import Lstm
from model_GRU import GRU
from model_gpt import GPT
import warnings
import sys
from data_utils import load_presplit_data_from_dir
warnings.filterwarnings("ignore", message="Implicit dimension choice for log_softmax has been deprecated.", category=UserWarning, module="torch.nn.modules.module")
# 假设你的 NMTCritierion 在 utilis_data.py 中
# 你需要确保这个导入路径是正确的，或者将 NMTCritierion 的定义复制过来

try:
    from utilis_data import NMTCritierion
    HAS_NMTCriterion = True
except ImportError:
    print("警告: 未找到 utilis_data.NMTCritierion。将使用 nn.CrossEntropyLoss。")
    HAS_NMTCriterion = False
    NMTCritierion = None # 占位

def parse_arguments():
    parser = argparse.ArgumentParser(description="深度学习模型训练脚本 (从预分割数据加载)")
    parser.add_argument('--presplit_data_dir', type=str, required=True,
                        help="包含预分割CSV文件的目录路径。")
    parser.add_argument('--label_column', type=str, default="dead", help="CSV文件中标签列的名称。")
    parser.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu', help='指定运行设备。')
    parser.add_argument('--seed', type=int, default=42, help='模型训练过程中的随机种子。') # 与原始脚本的seed一致
    
    parser.add_argument('--load_pretrained_model', type=str, default=None,
                    help="预训练模型文件 (.pth) 的路径。如果提供，则加载权重。")
    parser.add_argument('--freeze_backbone', action='store_true',
                    help="如果加载预训练模型，是否冻结骨干网络参数（除分类头外）。")
    
    parser.add_argument('--model_name', type=str, default="Lstm", choices=["Transformer", "Lstm", "GRU", "GPT"], help='选择模型架构。')
    # Transformer specific (Lstm用不到这些，但argparse会处理)
    parser.add_argument('--model_dim', type=int, default=128, help='Transformer/GPT 内部维度。')
    parser.add_argument('--depth', type=int, default=8, help='Transformer 深度。')
    parser.add_argument('--num_heads', type=int, default=8, help='Transformer/GPT 注意力头数。')
    
    # 通用或特定模型
    parser.add_argument('--drop_ratio', type=float, default=0.1, help='模型中的 Dropout 比率。')
    parser.add_argument('--num_layers', type=int, default=2, help="GRU/GPT 的层数 (Lstm层数是固定的)。") # Lstm不直接用这个
    
    parser.add_argument('--lr', type=float, default=5e-4, help='初始学习率。')
    parser.add_argument('--epochs', type=int, default=800, help='训练的总轮数。')
    parser.add_argument('--batch_size', type=int, default=5000, help='数据加载的批量大小。')
    
    parser.add_argument('--output_base_dir', type=str, default="./dl_model_outputs_replication",
                        help="保存输出的主目录。")
    parser.add_argument('--experiment_tag', type=str, default="", help="实验的额外标签。")
    opts = parser.parse_args()
    return opts

def initialize_model(args, num_features):
    # 首先，根据模型名称和参数实例化模型
    if args.model_name == "Lstm":
        print(f"初始化 Lstm模型: factors={num_features}, batch_size={args.batch_size}, drop_ratio={args.drop_ratio}, device={args.device}")
        model = Lstm(
            factors=num_features,
            batch_size=args.batch_size,
            drop_ratio=args.drop_ratio,
            device=args.device
        )
    elif args.model_name == "Transformer":
        print(f"初始化 Transformer模型: input_dim={num_features}, model_dim={args.model_dim}, depth={args.depth}, num_heads={args.num_heads}, drop_ratio={args.drop_ratio}")
        # 确保 Transformer 构造函数与你的 Transformer.py 一致
        model = Transformer(
            input_dim=num_features,
            model_dim=args.model_dim,
            depth=args.depth,
            num_heads=args.num_heads,
            drop_ratio=args.drop_ratio
            # qkv_bias=True, # 根据你的Transformer.py添加其他必要参数
            # attn_drop_ratio=args.drop_ratio,
            # drop_path_ratio=args.drop_path_ratio if hasattr(args, 'drop_path_ratio') else 0.1,
        )
    elif args.model_name == "GRU":
        print(f"初始化 GRU模型: factors={num_features}, batch_size={args.batch_size}, num_layers={args.num_layers}, drop_ratio={args.drop_ratio}, device={args.device}")
        model = GRU(
            factors=num_features,
            batch_size=args.batch_size,
            num_layers=args.num_layers,
            drop_ratio=args.drop_ratio,
            device=args.device
        )
    elif args.model_name == "GPT":
        # 严格遵循原始 train.py 中 GPT 的实例化逻辑
        gpt_actual_input_dim = num_features
        gpt_internal_hidden_dim = num_features # 模型内部工作维度，为了匹配原始 hidden_dim=factors
        gpt_internal_num_heads = 4           # 硬编码自原始 train.py
        gpt_internal_ff_expansion_factor = 128 # 硬编码自原始 train.py

        print(f"初始化 GPT模型 (尝试严格复现原始 train.py 实例化):")
        print(f"  actual_input_dim: {gpt_actual_input_dim}")
        print(f"  hidden_dim: {gpt_internal_hidden_dim}")
        print(f"  num_heads: {gpt_internal_num_heads}")
        print(f"  num_layers: {args.num_layers}")
        print(f"  dropout: {args.drop_ratio}")
        print(f"  ff_expansion_factor: {gpt_internal_ff_expansion_factor}")

        # 确保你的 model_gpt.py 中的 GPT 类 __init__ 签名是这样的：
        # def __init__(self, actual_input_dim, hidden_dim, num_heads, num_layers,
        #              ff_expansion_factor, dropout, max_seq_len=512, device='cuda'):
        # 并且它内部正确使用了 actual_input_dim 来创建一个投影层到 hidden_dim
        model = GPT(
            actual_input_dim=gpt_actual_input_dim,
            hidden_dim=gpt_internal_hidden_dim,
            num_heads=gpt_internal_num_heads,
            num_layers=args.num_layers,
            ff_expansion_factor=gpt_internal_ff_expansion_factor,
            dropout=args.drop_ratio,
            device=args.device
            # max_seq_len 如果需要，也应传递
        )
    else:
        raise ValueError(f"未知的模型名称: {args.model_name}")

    # 将模型移动到设备
    model = model.to(args.device)

    # --- 加载预训练权重逻辑 ---
    if args.load_pretrained_model and hasattr(args, 'load_pretrained_model') and args.load_pretrained_model is not None: # 检查参数是否存在且非None
        if os.path.exists(args.load_pretrained_model):
            print(f"\n加载预训练权重从: {args.load_pretrained_model}")
            try:
                pretrained_state_dict = torch.load(args.load_pretrained_model, map_location=args.device)

                # 处理常见的state_dict包装方式
                if isinstance(pretrained_state_dict, dict):
                    if 'state_dict' in pretrained_state_dict:
                        pretrained_state_dict = pretrained_state_dict['state_dict']
                    elif 'model_state_dict' in pretrained_state_dict:
                        pretrained_state_dict = pretrained_state_dict['model_state_dict']
                    # 如果直接就是 state_dict，则无需处理

                current_model_dict = model.state_dict()

                # --- 确定分类头的名称 ---
                # 这需要根据你每个模型的具体实现来确定
                # 假设 Transformer 的分类头是 'MLP' (基于你原始的 Transformer.py 中的 self.MLP)
                # 假设 Lstm, GRU, GPT 的分类头都叫做 'head' (基于常见的命名)
                # 你需要根据实际情况修改这些名称！
                classifier_head_keyword = None
                if args.model_name == "Transformer":
                    classifier_head_keyword = "MLP." # 加点号确保是模块名的一部分
                elif args.model_name in ["Lstm", "GRU", "GPT"]:
                    # 检查你的模型定义中分类头的真实名称
                    # 假设在这些模型中，最终的线性层或包含最终线性层的Sequential块叫做 'head'
                    if hasattr(model, 'head'):
                        classifier_head_keyword = "head."
                    else:
                        print(f"警告: 模型 {args.model_name} 没有找到名为 'head' 的属性，无法准确排除分类头权重。请检查模型定义。")
                
                if classifier_head_keyword is None and args.model_name != "Transformer": # Transformer 有特例
                     print(f"警告: 未能确定模型 {args.model_name} 的分类头名称，将尝试加载所有匹配的权重。这可能导致分类头也被预训练权重覆盖。")


                state_dict_to_load = {}
                weights_loaded_count = 0
                weights_skipped_shape_mismatch = 0
                weights_skipped_classifier = 0

                for k, v_pretrained in pretrained_state_dict.items():
                    load_this_weight = True
                    if classifier_head_keyword: # 如果我们定义了分类头的关键词
                        if k.startswith(classifier_head_keyword):
                            weights_skipped_classifier += 1
                            load_this_weight = False # 不加载分类头的权重
                    
                    if load_this_weight:
                        if k in current_model_dict:
                            v_current = current_model_dict[k]
                            if v_current.shape == v_pretrained.shape:
                                state_dict_to_load[k] = v_pretrained
                                weights_loaded_count += 1
                            else:
                                print(f"  形状不匹配，跳过加载权重 {k}: 预训练 {v_pretrained.shape}, 当前 {v_current.shape}")
                                weights_skipped_shape_mismatch += 1
                        # else: # 预训练模型中有，但当前模型没有的键，通常可以忽略
                        #     print(f"  权重 {k} 在预训练模型中存在，但在当前模型中不存在，跳过。")
                
                if not state_dict_to_load:
                    print("警告: 没有从预训练模型中找到可加载的匹配权重（非分类头部分）。")
                else:
                    print(f"将从预训练模型加载 {weights_loaded_count} 个参数组。")
                    if weights_skipped_classifier > 0:
                        print(f"  跳过了 {weights_skipped_classifier} 个分类头相关的参数组。")
                    if weights_skipped_shape_mismatch > 0:
                        print(f"  因形状不匹配跳过了 {weights_skipped_shape_mismatch} 个参数组。")
                    
                    current_model_dict.update(state_dict_to_load)
                    # 使用 strict=False 允许当前模型中存在预训练模型没有的键（例如新初始化的分类头）
                    # 也允许预训练模型中存在当前模型没有的键（虽然我们已经通过 k in current_model_dict 过滤了一部分）
                    model.load_state_dict(current_model_dict, strict=False)
                    print("预训练权重加载完成。分类头（如果存在且未被加载）保持其原有初始化。")

                # --- 冻结骨干网络参数 ---
                if args.freeze_backbone and hasattr(args, 'freeze_backbone') and args.freeze_backbone:
                    print("冻结骨干网络参数...")
                    num_frozen = 0
                    num_trainable = 0
                    for name, param in model.named_parameters():
                        freeze_this_param = True
                        if classifier_head_keyword: # 如果我们定义了分类头的关键词
                            if name.startswith(classifier_head_keyword):
                                freeze_this_param = False # 不冻结分类头的参数
                        
                        if freeze_this_param:
                            param.requires_grad = False
                            num_frozen +=1
                        else:
                            param.requires_grad = True # 确保分类头是可训练的
                            print(f"  分类头参数 '{name}' 将保持可训练。")
                            num_trainable +=1
                    print(f"  已冻结 {num_frozen} 组参数。")
                    print(f"  保持可训练 {num_trainable} 组参数 (主要是分类头)。")
            except Exception as e:
                print(f"错误: 加载预训练权重时发生异常: {e}")
                print("将从头开始训练模型。")
        else:
            print(f"警告: 预训练模型文件未找到: {args.load_pretrained_model}。将从头开始训练。")
    else:
        print("未提供预训练模型路径，或未启用预训练加载。模型将从头开始训练。")
        
    return model

def prepare_input_for_model(batch_data, model_name, device):
    # 这个函数与原始 _prepare_input 逻辑一致
    input_features = batch_data['data'].to(device, non_blocking=True)
    targets = batch_data['target'].to(device, non_blocking=True)
    if input_features.ndim == 2:
        input_features = input_features.unsqueeze(1)
    if model_name in ["Lstm", "GRU"]:
        if input_features.shape[1] == 1:
            input_features = input_features.permute(1, 0, 2)
    return input_features, targets

def main():
    args = parse_arguments()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    print("\n--- 参数配置 ---")
    for arg, value in sorted(vars(args).items()):
        print(f"  {arg}: {value}")
    print("-" * 30)

    current_time = time.strftime("%Y%m%d-%H%M%S")
    data_source_tag = os.path.basename(os.path.normpath(args.presplit_data_dir))
    experiment_name_parts = [args.model_name, data_source_tag.replace('_split','')]
    if args.experiment_tag: experiment_name_parts.append(args.experiment_tag)
    experiment_name_parts.append(f"trainseed{args.seed}")
    experiment_name_parts.append(current_time)
    experiment_run_name = "_".join(experiment_name_parts)
    experiment_output_dir = os.path.join(args.output_base_dir, experiment_run_name)
    os.makedirs(experiment_output_dir, exist_ok=True)
    print(f"所有输出将保存到: {experiment_output_dir}")

    train_loader, test_loader, n_train, n_test, num_features = load_presplit_data_from_dir(
        presplit_data_dir=args.presplit_data_dir, batch_size=args.batch_size,
        label_column=args.label_column
    )
    print(f"成功从预分割目录加载数据: {args.presplit_data_dir}，特征数: {num_features}")

    model = initialize_model(args, num_features)
    print("\n--- 模型结构 ---")
    print(model)
    print("-" * 30)

    # 严格按照原始 train.py 的优化器设置
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=0) # 原始代码 weight_decay=0
    print(f"优化器: Adam, LR: {args.lr}, Weight Decay: 0")

    # 严格按照原始 train.py 的损失函数设置
    if HAS_NMTCriterion:
        label_smoothing_value = 0.3
        criterion = NMTCritierion(label_smoothing=label_smoothing_value).to(args.device)
        print(f"损失函数: NMTCritierion (标签平滑: {label_smoothing_value})")
    else:
        criterion = nn.CrossEntropyLoss().to(args.device)
        print(f"损失函数: nn.CrossEntropyLoss (因为 NMTCritierion 未找到)")


    best_test_auc = 0.0
    best_model_epoch = -1
    final_test_labels_for_delong = None
    final_test_probs_for_delong = None

    print("\n--- 开始训练 ---")
    for epoch in range(args.epochs):
        epoch_start_time = time.time()
        model.train()
        total_train_loss = 0
        for batch_idx, batch in enumerate(train_loader):
            input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
            optimizer.zero_grad()
            outputs = model(input_features) # outputs 是概率 (因为Lstm模型末尾有Softmax)

            
            if isinstance(criterion, nn.CrossEntropyLoss):
              
                if HAS_NMTCriterion and isinstance(criterion, NMTCritierion):
                    loss = criterion(outputs, targets) # 假设 NMTCritierion 能处理概率或内部有log
                elif isinstance(criterion, nn.CrossEntropyLoss):
                    # 这里是个问题点：CELoss期望logits，但Lstm输出概率
                    # 我们需要 log(probabilities) 作为 CELoss 的近似输入（但不完全等同于logits）
                    # 或者，更标准的是，Lstm模型不应有最后的Softmax
                    print("警告: Lstm模型输出概率，但nn.CrossEntropyLoss期望logits。结果可能不准确。建议修改Lstm模型移除末尾Softmax。")
                    loss = criterion(torch.log(outputs + 1e-9), targets) # 加一个小值避免log(0)
                else: # 其他自定义损失
                    loss = criterion(outputs, targets)

            else: # 默认情况，比如就是 NMTCritierion
                 loss = criterion(outputs, targets)


            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
        avg_train_loss = total_train_loss / len(train_loader)

        model.eval()
        current_epoch_test_targets_list = []
        current_epoch_test_probs_list = []

        with torch.no_grad():
            for batch in test_loader:
                input_features, targets = prepare_input_for_model(batch, args.model_name, args.device)
                outputs_probs = model(input_features) # Lstm 直接输出概率

                positive_class_probs = outputs_probs[:, 1]
                current_epoch_test_targets_list.extend(targets.cpu().numpy())
                current_epoch_test_probs_list.extend(positive_class_probs.cpu().numpy())

        epoch_targets_np = np.array(current_epoch_test_targets_list)
        epoch_probs_np = np.array(current_epoch_test_probs_list)
        epoch_preds_np = (epoch_probs_np >= 0.5).astype(int)

        test_acc = accuracy_score(epoch_targets_np, epoch_preds_np)
        test_auc = 0.0
        try:
            if len(np.unique(epoch_targets_np)) > 1 and len(epoch_probs_np) > 0:
                test_auc = roc_auc_score(epoch_targets_np, epoch_probs_np)
            else:
                print(f"警告: Epoch {epoch+1} 测试集标签种类不足或概率为空, AUC设为0.0。")
        except ValueError as e:
            print(f"警告: Epoch {epoch+1} 计算AUC时出错 ({e}), AUC设为0.0。")

        epoch_duration = time.time() - epoch_start_time


        if test_auc > best_test_auc:
            best_test_auc = test_auc
            best_model_epoch = epoch + 1
            final_test_labels_for_delong = epoch_targets_np.copy()
            final_test_probs_for_delong = epoch_probs_np.copy()
            torch.save(model.state_dict(), os.path.join(experiment_output_dir, "best_auc_model.pth"))
            print(f"  * New best Test AUC: {best_test_auc:.4f} at Epoch {best_model_epoch}. Model and DeLong data updated.")
            print(f"Epoch [{epoch+1:03d}/{args.epochs:03d}] | "
              f"Train Loss: {avg_train_loss:.4f} | Test Acc: {test_acc:.4f} | "
              f"Test AUC: {test_auc:.4f} | Time: {epoch_duration:.2f}s")
    print("\n--- 训练完成 ---")
    # ... (后续的保存DeLong文件和参数摘要逻辑与之前版本相同) ...
    if best_model_epoch != -1:
        print(f"最佳测试集 AUC: {best_test_auc:.4f} (在 Epoch {best_model_epoch} 获得)")
        if final_test_labels_for_delong is not None and final_test_probs_for_delong is not None:
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_labels_best_auc.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, f"{args.model_name}_test_probs_best_auc.npy"), final_test_probs_for_delong)
            print(f"为DeLong检验准备的最终测试集标签和概率已保存.")
            np.save(os.path.join(experiment_output_dir, "test_labels.npy"), final_test_labels_for_delong)
            np.save(os.path.join(experiment_output_dir, "test_probabilities.npy"), final_test_probs_for_delong)
            print(f"同时保存为通用文件名: test_labels.npy, test_probabilities.npy")
    else:
        print("没有在训练过程中记录到有效的最佳AUC模型。")

    args_dict = vars(args)
    args_dict['num_input_features'] = num_features
    args_dict['best_test_auc_achieved'] = best_test_auc
    args_dict['best_model_epoch'] = best_model_epoch
    args_df = pd.DataFrame([args_dict])
    args_df.to_csv(os.path.join(experiment_output_dir, "training_args_summary.csv"), index=False)
    print(f"训练参数和摘要已保存。")



import sys # 确保导入sys模块


if __name__ == '__main__':
    # --- 预训练模型路径 ---
    PRETRAINED_GPT_MODEL_PATH = "./dl_model_outputs_replication/GPT_mimic3_eICU_8_factors_mimic3eICU-8f-gpt-replication_trainseed42_20250610-051542/best_auc_model.pth"

    # --- 模拟命令行参数: 测试 GPT 在 8因子 MIMIC-IV 数据上 (带预训练) ---
    sys.argv = [
        'my_script_name_in_notebook.py', # 脚本名占位符
        '--presplit_data_dir', './temp_data_utils_logs/mimic4_8_factors_split', # 8因子 MIMIC-IV 预分割数据目录
        '--model_name', 'GPT',                                               # 模型名称
        # --- GPT 特定参数 (将由 initialize_model 根据原始逻辑处理) ---
        '--num_layers', '1',        # GPT 的层数 (与原始摘要一致)
        '--drop_ratio', '0.1',      # Dropout 比率 (与原始摘要一致)
        # --- 通用参数 (严格参考你提供的原始摘要) ---
        '--lr', '0.005',            # 学习率
        '--epochs', '6000',         # Epochs
        '--batch_size', '5000',     # 批量大小
        '--output_base_dir', './dl_model_outputs_replication', # 输出基础目录
        '--experiment_tag', 'mimic4-8f-gpt-finetune',        # 实验标签，指明是微调
        '--seed', '42',             # 随机种子
        '--device', 'cuda',         # 如果可用，使用cuda
        '--label_column', 'dead',   # 标签列名
        # --- 新增预训练参数 ---
        '--load_pretrained_model', PRETRAINED_GPT_MODEL_PATH,
        # '--freeze_backbone' # 如果需要冻结骨干，取消此行注释
        # --- 以下参数如果命令行中提供了，initialize_model for GPT 会有特殊处理以复现原始行为 ---
        # '--model_dim', '128', # GPT的hidden_dim会用num_features (8)
        # '--num_heads', '8',   # GPT的num_heads会硬编码为4
    ]
    # --------------------------------------------------------------------

    # 调用你的主函数
    main()



--- 参数配置 ---
  batch_size: 5000
  depth: 8
  device: cuda
  drop_ratio: 0.1
  epochs: 6000
  experiment_tag: mimic4-8f-gpt-finetune
  freeze_backbone: False
  label_column: dead
  load_pretrained_model: ./dl_model_outputs_replication/GPT_mimic3_eICU_8_factors_mimic3eICU-8f-gpt-replication_trainseed42_20250610-051542/best_auc_model.pth
  lr: 0.005
  model_dim: 128
  model_name: GPT
  num_heads: 8
  num_layers: 1
  output_base_dir: ./dl_model_outputs_replication
  presplit_data_dir: ./temp_data_utils_logs/mimic4_8_factors_split
  seed: 42
------------------------------
所有输出将保存到: ./dl_model_outputs_replication/GPT_mimic4_8_factors_mimic4-8f-gpt-finetune_trainseed42_20250610-055715
从预分割文件加载数据:
  训练数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_train_data_seed256.csv
  测试数据: ./temp_data_utils_logs/mimic4_8_factors_split/split_test_data_seed256.csv
  - 特征数量: 8
  - 训练样本数: 4525
  - 测试样本数: 1940
成功从预分割目录加载数据: ./temp_data_utils_logs/mimic4_8_factors_split，特征数: 8
初始化 GPT模型 (尝试严格复现原始 trai

/tmp/ipykernel_754/586191678.py:135: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_state_dict = torch.load(args.load_pretrained_model, map_location=args.device)


优化器: Adam, LR: 0.005, Weight Decay: 0
损失函数: NMTCritierion (标签平滑: 0.3)

--- 开始训练 ---


/mnt/public/home/zhijiangwan/.local/lib/python3.8/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='sum' instead.
  warnings.warn(warning.format(ret))


  * New best Test AUC: 0.4981 at Epoch 1. Model and DeLong data updated.
Epoch [001/6000] | Train Loss: 603.1953 | Test Acc: 0.5000 | Test AUC: 0.4981 | Time: 0.95s
  * New best Test AUC: 0.5138 at Epoch 2. Model and DeLong data updated.
Epoch [002/6000] | Train Loss: 506.2319 | Test Acc: 0.5031 | Test AUC: 0.5138 | Time: 0.12s
  * New best Test AUC: 0.5366 at Epoch 3. Model and DeLong data updated.
Epoch [003/6000] | Train Loss: 448.8775 | Test Acc: 0.5031 | Test AUC: 0.5366 | Time: 0.26s
  * New best Test AUC: 0.5378 at Epoch 257. Model and DeLong data updated.
Epoch [257/6000] | Train Loss: 280.7552 | Test Acc: 0.4716 | Test AUC: 0.5378 | Time: 0.11s
  * New best Test AUC: 0.5396 at Epoch 312. Model and DeLong data updated.
Epoch [312/6000] | Train Loss: 276.0717 | Test Acc: 0.4701 | Test AUC: 0.5396 | Time: 0.11s
  * New best Test AUC: 0.5403 at Epoch 313. Model and DeLong data updated.
Epoch [313/6000] | Train Loss: 276.0116 | Test Acc: 0.4680 | Test AUC: 0.5403 | Time: 0.11s
  * 